# ECE 471 Sample Project



In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a project folder for EfficientAD weights
WEIGHTS_PATH = '/content/drive/MyDrive/ECE471_Project/weights'
os.makedirs(WEIGHTS_PATH, exist_ok=True)

print(f"Weights will be saved to: {WEIGHTS_PATH}")

Mounted at /content/drive
Weights will be saved to: /content/drive/MyDrive/ECE471_Project/weights


In [ ]:
# Download the dataset, setup packages
import os
import cv2
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score

if not os.path.exists('dataset.zip'):
  !gdown 1_pRKXtYRjWjY0seYqyx25nOxjtr-mHYg
  !unzip -q -u dataset.zip
else:
  print('Already downloaded')

Downloading...
From: https://drive.google.com/uc?id=1_pRKXtYRjWjY0seYqyx25nOxjtr-mHYg
To: /content/dataset.zip
100% 2.60M/2.60M [00:00<00:00, 259MB/s]


In [ ]:
# Some helper functions for your project
def load_dataset(class_name = 'pasta'):
  assert class_name in ['pasta', 'screws', 'capsule']
  dir = './dataset/'+class_name+'/'
  training_images = []
  testing_images = []
  testing_labels = []
  for file_name in os.listdir(dir+'train/good/'):
    training_images.append(cv2.cvtColor(cv2.imread(dir+'train/good/'+file_name), cv2.COLOR_BGR2RGB))
  for file_name in os.listdir(dir+'test/good/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/good/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(0)
  for file_name in os.listdir(dir+'test/bad/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/bad/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(1)

  # returns a normalized (0-1) numpy array of size (n,)
  return np.array(training_images)/255., np.array(testing_images)/255., np.array(testing_labels)

def basic_evaluation(predictions : np.ndarray, targets : np.ndarray):
  print(targets)
  print(predictions)
  print('AUROC Score:', roc_auc_score(targets, predictions))

In [ ]:
# TODO import any packages you would like to use here
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision
from fastai.data.external import untar_data, URLs
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
import torch.optim as optim
torch.backends.cudnn.benchmark = True

In [ ]:
# Load the pretrained weights of WideResNet-101 trained on ImageNet
# This will act as the truth from which the teacher network will learn
oracle = models.wide_resnet101_2(weights='IMAGENET1K_V1')
# Set to evaluation mode as we are not training it, just using it.
oracle.eval()

# Freeze all of the oracle's weight values
for param in oracle.parameters():
  param.requires_grad = False

# Switch to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

oracle.to(device)

print(f"WideResNet-101 loaded and frozen on {device}.")

Downloading: "https://download.pytorch.org/models/wide_resnet101_2-32ee1156.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet101_2-32ee1156.pth


100%|██████████| 243M/243M [00:01<00:00, 239MB/s]


WideResNet-101 loaded and frozen on cuda.


In [6]:
# Get the Imagenette images
# Imagenette is a labelled subset of ImagNet
path = untar_data(URLs.IMAGENETTE)

distillation_data_path = os.path.join(path, 'train')

<div><progress max="1557161267" value="1557168128"></progress> 100.00% [1557168128/1557161267 02:00&lt;00:00]</div>

In [7]:
class OracleWrapper(nn.Module):
  def __init__(self, backbone):
    super(OracleWrapper, self).__init__()
    self.backbone = backbone

    self.layer2 = backbone.layer2
    self.layer3 = backbone.layer3

  def forward(self, x):

    # Initialize layers of WideResNet
    x = self.backbone.conv1(x)
    x = self.backbone.bn1(x)
    x = self.backbone.relu(x)
    x = self.backbone.maxpool(x)
    x = self.backbone.layer1(x)

    # We want to extract intermediate features as done in the EfficientAD paper
    f2 = self.layer2(x)
    f3 = self.layer3(f2)

    # Resize such that both layers are 64x64 so they can be combined
    f2 = F.interpolate(f2, size=(64, 64), mode='bilinear', align_corners=False)
    f3 = F.interpolate(f3, size=(64, 64), mode='bilinear', align_corners=False)
    # Combine the 512 and 1024 channels into one block of size 1536
    combined = torch.cat([f2, f3], dim=1)
    # Project this down to 384 channels as required by EfficientAD
    # We take the first 384 channels as these should have the most information
    projected = combined[:, :384, :, :]

    return projected

oracle_extractor = OracleWrapper(oracle).to(device)




In [8]:
class InMemoryDataset(Dataset):
    def __init__(self, base_dataset):
        self.images = []
        self.labels = []
        print("Pre-loading dataset into RAM.")
        for img, label in base_dataset:
            self.images.append(img)
            self.labels.append(label)
        # Stack into tensors upfront for faster indexing
        self.images = torch.stack(self.images)
        self.labels = torch.tensor(self.labels)
        print(f"Loaded {len(self.images)} images into RAM.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

base_dataset = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=transform)
ram_dataset = InMemoryDataset(base_dataset)

fast_loader = DataLoader(
    ram_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=True
)

Pre-loading dataset into RAM.
Loaded 9469 images into RAM.


In [ ]:
num_images_to_sample = 500
channels = 384
# Trackers for running sum and sum of squares
sum_features = torch.zeros(channels, device=device)
sum_sq_features = torch.zeros(channels, device=device)
total_pixels = 0
oracle_extractor.eval()
with torch.no_grad():
    for i, (images, _) in enumerate(fast_loader):
        # We process in batches of 16 now, so break when we hit our target
        if i * fast_loader.batch_size >= num_images_to_sample:
            break

        images = images.to(device, non_blocking=True)

        # We need 512x512 for the oracle (Algorithm 3 context)
        images_oracle = F.interpolate(images, size=(512, 512), mode='bilinear', align_corners=False)
        features = oracle_extractor(images_oracle) # Shape: [B, 384, 64, 64]

        # Number of spatial pixels per batch: B * H * W
        B, C, H, W = features.shape
        pixels_in_batch = B * H * W
        total_pixels += pixels_in_batch

        # Sum over Batch, Height, and Width (dims 0, 2, 3)
        sum_features += features.sum(dim=(0, 2, 3))
        sum_sq_features += (features ** 2).sum(dim=(0, 2, 3))
        if (i * fast_loader.batch_size) % 100 < fast_loader.batch_size:
            print(f"Processed ~{i * fast_loader.batch_size}/{num_images_to_sample} images for stats...")
# E[X] = sum(X) / N
oracle_mean = sum_features / total_pixels
# Var(X) = E[X^2] - (E[X])^2
oracle_variance = torch.sqrt((sum_sq_features / total_pixels) - (oracle_mean ** 2))
print(f"Mean shape: {oracle_mean.shape}")
print(f"Variance shape: {oracle_variance.shape}")
torch.save({'mu': oracle_mean, 'sigma': oracle_variance},
           os.path.join(WEIGHTS_PATH, 'oracle_stats.pth'))



In [ ]:
class AutoEncoder(nn.Module):
  def __init__(self,):
    super().__init__()

    # Encoder Layers as class attributes. Conv2d docs:
    # https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
    # Input channels is what the layer receives, and output channels is
    # specified in table 8 of the paper.
    # Kernel size, stride, and padding all taken directly from table 8
    self.EncConv1 = nn.Conv2d(in_channels=3, out_channels=32,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv2 = nn.Conv2d(in_channels=32, out_channels=32,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv3 = nn.Conv2d(in_channels=32, out_channels=64,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv4 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv5 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(2, 2), padding=1)
    self.EncConv6 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(8, 8), stride=(1, 1), padding=0)

    # Decoder layers (more Conv2D)
    self.DecConv1 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv2 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv3 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv4 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv5 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv6 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(4, 4), stride=(1, 1), padding=2)
    self.DecConv7 = nn.Conv2d(in_channels=64, out_channels=64,
                              kernel_size=(3, 3), stride=(1, 1), padding=1)
    self.DecConv8 = nn.Conv2d(in_channels=64, out_channels=384,
                              kernel_size=(3, 3), stride=(1, 1), padding=1)

    # Bilinear Layers: https://docs.pytorch.org/docs/stable/generated/torch.nn.Upsample.html
    self.Bilinear1 = nn.Upsample(size=(3, 3), mode="bilinear")
    self.Bilinear2 = nn.Upsample(size=(8, 8), mode="bilinear")
    self.Bilinear3 = nn.Upsample(size=(15, 15), mode="bilinear")
    self.Bilinear4 = nn.Upsample(size=(32, 32), mode="bilinear")
    self.Bilinear5 = nn.Upsample(size=(63, 63), mode="bilinear")
    self.Bilinear6 = nn.Upsample(size=(127, 127), mode="bilinear")
    self.Bilinear7 = nn.Upsample(size=(64, 64), mode="bilinear")

    # Dropout Layers: https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html
    self.Dropout1To6 = nn.Dropout(p=0.2) # All dropout layers have the same p value

    # Relu activation function (for convenience)
    self.relu = nn.ReLU()

  def forward(self, x):
    # Compute a forward pass. Here we just combine the layers in the order of table 8 in the EfficientAD paper
    out = self.relu(self.EncConv1(x))
    out = self.relu(self.EncConv2(out))
    out = self.relu(self.EncConv3(out))
    out = self.relu(self.EncConv4(out))
    out = self.relu(self.EncConv5(out))
    out = self.EncConv6(out)

    out = self.Bilinear1(out)

    out = self.relu(self.DecConv1(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear2(out)

    out = self.relu(self.DecConv2(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear3(out)

    out = self.relu(self.DecConv3(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear4(out)

    out = self.relu(self.DecConv4(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear5(out)

    out = self.relu(self.DecConv5(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear6(out)

    out = self.relu(self.DecConv6(out))
    out = self.Dropout1To6(out)
    out = self.Bilinear7(out)

    out = self.relu(self.DecConv7(out))
    out = self.DecConv8(out)

    return out

In [11]:
class PDN(nn.Module):
  def __init__(self, out_channels=384, padding=True):
    super(PDN, self).__init__()

    p = 3 if padding else 0

    self.conv1 = nn.Conv2d(3, 128, kernel_size=4, stride=1, padding=p);
    self.avgpool1 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1);
    self.conv2 = nn.Conv2d(128, 256, kernel_size=4, stride=1, padding=p);
    self.avgpool2 = nn.AvgPool2d(kernel_size=2, stride=2, padding=1);
    self.conv3 = nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1);
    self.conv4 = nn.Conv2d(256, out_channels, kernel_size=4, stride=1, padding=0);

    self.relu = nn.ReLU(inplace=True);

    # Compute the forward pass
  def forward(self, x):
    x = self.relu(self.conv1(x))
    x = self.avgpool1(x)
    x = self.relu(self.conv2(x))
    x = self.avgpool2(x)
    x = self.relu(self.conv3(x))
    x = self.conv4(x)
    return x

In [ ]:
# We use this helper to calculate the mean squared error
# This method is described in Section 3.1 and Algorithm 3 of the EfficientAD paper
def distillation_loss(teacher_output, oracle_output, mu, sigma):

  # Reshape mu and sigma
  mu = mu.view(1, -1, 1, 1)
  sigma = sigma.view(1, -1, 1, 1)
  # Normalize the oracle output
  normalized_oracle = (oracle_output - mu) / sigma
  # Compute the loss
  loss = torch.mean((teacher_output - normalized_oracle) ** 2)

  return loss

In [13]:
def train_teacher(teacher, oracle_extractor, distill_images, oracle_mean, oracle_variance, iterations=60000, batch_size=16):
  teacher.train()
  oracle_extractor.eval()
  # Algorithm 3, Line 13: Adam with lr=1e-4 and weight_decay=1e-5
  optimizer = optim.Adam(teacher.parameters(), lr=1e-4, weight_decay=1e-5)
  # Normalize oracle output (Algorithm 3, Line 22)
  mu = oracle_mean.view(1, -1, 1, 1).to(device)
  sigma = oracle_variance.view(1, -1, 1, 1).to(device)
  # Algorithm 3, Lines 14-30: Main training loop
  for iteration in range(1, iterations + 1):
    # Algorithm 3, Lines 16-27: Inner batch loop of 16 images
    # We dont actually need to loop like is done in the algorithm, can process the whole batch at once
    indices = torch.randint(0, len(distill_images), (batch_size,), device=distill_images.device)
    images = distill_images[indices]

    # Algorithm 3, Line 18: Convert to grayscale with probability 0.1
    # Apply this to batch all at once using a mask
    gray = transforms.functional.rgb_to_grayscale(images, num_output_channels=3)
    gray_mask = (torch.rand(images.shape[0], 1, 1,1, device=device) < 0.1)
    images = torch.where(gray_mask, gray, images)

    # Algorithm 3, Line 19: Images are already resized to 256x256 by the dataloader
    # but we need a 512x512 version for the oracle
    images_oracle = F.interpolate(images, size=(512, 512), mode='bilinear', align_corners=False)

    # Algorithm 3, Lines 21-22: Get normalized oracle features
    with torch.no_grad():
      y_oracle = oracle_extractor(images_oracle)
      y_oracle_normalized = (y_oracle - mu) / sigma

    # Algorithm 3, Line 23: Get teacher features on 256x256 image
    y_teacher = teacher(images)

    # Algorithm 3, Lines 24-28: Compute squared difference and mean loss
    D_dist = (y_oracle_normalized - y_teacher) ** 2
    L_batch = D_dist.mean()
    # End of Pseudocode loop over batch

    # Algorithm 3, Line 29: Update teacher parameters
    optimizer.zero_grad()
    L_batch.backward()
    optimizer.step()
    if iteration % 500 == 0:
      print(f"Iteration {iteration}/{iterations} | Loss: {L_batch.item():.6f}")


In [ ]:
# Before calling train_teacher, extract just the images tensor from ram_dataset
distill_images = ram_dataset.images.to(device)  # Move entire dataset to GPU once
teacher = PDN(out_channels=384).to(device)
train_teacher(teacher, oracle_extractor, distill_images, oracle_mean, oracle_variance)
torch.save(teacher.state_dict(), os.path.join(WEIGHTS_PATH, 'teacher_FINAL.pth'))
print("TEACHER WEIGHTS SAVED RAAAAAH")

Iteration 500/60000 | Loss: 0.644774
Iteration 1000/60000 | Loss: 0.592735
Iteration 1500/60000 | Loss: 0.623805
Iteration 2000/60000 | Loss: 0.605449
Iteration 2500/60000 | Loss: 0.589505
Iteration 3000/60000 | Loss: 0.466258
Iteration 3500/60000 | Loss: 0.464918
Iteration 4000/60000 | Loss: 0.563908
Iteration 4500/60000 | Loss: 0.526134
Iteration 5000/60000 | Loss: 0.478902
Iteration 5500/60000 | Loss: 0.453636
Iteration 6000/60000 | Loss: 0.436595
Iteration 6500/60000 | Loss: 0.410784
Iteration 7000/60000 | Loss: 0.441612
Iteration 7500/60000 | Loss: 0.473271
Iteration 8000/60000 | Loss: 0.418606
Iteration 8500/60000 | Loss: 0.401458
Iteration 9000/60000 | Loss: 0.415323
Iteration 9500/60000 | Loss: 0.419632
Iteration 10000/60000 | Loss: 0.369662
Iteration 10500/60000 | Loss: 0.394625
Iteration 11000/60000 | Loss: 0.453676
Iteration 11500/60000 | Loss: 0.417430
Iteration 12000/60000 | Loss: 0.405204
Iteration 12500/60000 | Loss: 0.341074
Iteration 13000/60000 | Loss: 0.404397
Iterat

In [14]:
PROJECT_WEIGHTS_PATH = '/content/drive/MyDrive/ECE471_Project/weights'
file_name = 'teacher_FINAL.pth'
full_save_path = os.path.join(PROJECT_WEIGHTS_PATH, file_name)
teacher = PDN(out_channels=384).to(device)
teacher.load_state_dict(torch.load(full_save_path))
teacher.eval()
for param in teacher.parameters():
  param.requires_grad = False

In [ ]:
# Possibly add student and autoencoder definitions here to get them without running training

In [ ]:
def HWC_to_NCHW(img, augment=False):
  img = torch.tensor(img, dtype=torch.float32)
  img = img.permute(2, 0, 1) # Reorder dims
  img = transforms.Resize((256, 256))(img) # Match expected input of the network
  if augment:
    augment_batch(img)
  img = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img) # Match mean and stdev of ImageNet
  img = img.unsqueeze(0) # Add batch dim (N)
  return img

In [ ]:
def image_preprocessing(images_np, device):
  img = torch.tensor(images_np, dtype=torch.float32)
  img = img.permute(0, 3, 1, 2).contiguous()
  img = torch.nn.functional.interpolate(img, size=(256, 256), mode='bilinear', align_corners=False)
  img = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(img)

  return img.to(device)

In [17]:
# Not currently used, but added in case we want it
def augment_batch(x):
  # Horizontal flip
  if np.random.random() > 0.5:
      img = transforms.functional.hflip(img)
  # Vertical flip
  if np.random.random() > 0.5:
      img = transforms.functional.vflip(img)
  # Minor rotation (+/- 10 degrees) to handle slight misalignment
  angle = np.random.uniform(-10, 10)
  img = transforms.functional.rotate(img, angle)

In [18]:
# Follow Algorithm 1 in the paper to train the autoencoder and student
# As in the paper, need pretrained teacher (we have), sequence of training/validation images (given)

def train_student_and_autoencoder(training_data, validation_data, testing_data, testing_labels):

  # 1. Compute Mu and Sigma based on training data set (Algorithm 1, Steps 3-9)
  teacher_outputs = []
  with torch.no_grad():
    for i in range(len(training_data)):
      # Prepare image and get teacher features
      img = training_data[i:i+1]
      y_prime = teacher(img) # Shape: [1, 384, 64, 64]
      teacher_outputs.append(y_prime)

  # Concatenate all outputs into one large tensor [N, 384, 64, 64]
  all_outputs = torch.cat(teacher_outputs, dim=0)

  # Compute mu and sigma per channel across the entire dataset
  # We reduce over Dim 0 (Images), Dim 2 (Height), and Dim 3 (Width)
  mu = torch.mean(all_outputs, dim=(0, 2, 3), keepdim=True)
  sigma = torch.std(all_outputs, dim=(0, 2, 3), keepdim=True)

  # Now we initialize a joint Adam optimizer for parameters of student and autoencoder (learning rate and weight decay from algorithm 1)
  lr = 5e-3
  weight_decay = 1e-5
  sa_parameters = list(student.parameters()) + list(autoencoder.parameters())
  sa_optimizer = optim.Adam(sa_parameters, lr=lr, weight_decay=weight_decay)

  # Initialize pretraining iterator once, outside the main loop.
  # We don't want to create a new pretraining iterator instance with each loop iteration
  distill_dataset = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=transform)
  distill_loader = DataLoader(distill_dataset, batch_size=1, shuffle=True) # add this because it is loading whole thing in teacher now
  pretrain_iter = iter(distill_loader)

  batch_size = min(8, len(training_data))

  # Main training loop
  for i in range(1, 7001):
    # Choose 'random' training image, do fwd pass with teacher, do fwd pass with student.
    indices = torch.randint(0, len(training_data), (batch_size,), device=training_data.device)
    random_image = training_data[indices]

    with torch.no_grad():
      teacher_output = teacher(random_image)
    student_output = student(random_image)

    # Get the normalized teacher output per channel
    normalized_teacher_output = (teacher_output - mu) / sigma

    # Now compute square difference between normalized teacher output and the first 384 entries of the student output
    student_output_f384 = student_output[:, :384, :, :]
    diff_st = (normalized_teacher_output - student_output_f384)**2

    # Get the 0.999 quantile of the difference, and compute loss as mean of diff_st entries over or equal to the quantile
    diff_st_999 = torch.quantile(torch.flatten(diff_st), q=0.999)

    # Pytorch boolean indexing rather than slow python loop
    loss_hard = diff_st[diff_st >= diff_st_999].mean()

    # Now choose a random pretraining ImageNet image. Load pretraining images more efficiently by
    # creating an infinite loop of the image data (so that we never run out) with a try block
    # This way we don't need to create a new pretraining iterator with each loop iteration
    try:
      pretraining_image, _ = next(pretrain_iter)
    except StopIteration:
      pretrain_iter = iter(distill_loader)
      pretraining_image, _ = next(pretrain_iter)
    pretraining_image = pretraining_image.to(device)

    # Now get the student teacher loss with hard loss and a regularization term defined in the algorithm
    student_output_pretaining_384 = student(pretraining_image)[:, :384, :, :]
    reg_term = (student_output_pretaining_384**2).mean()
    loss_st = loss_hard + reg_term

    # Now we augment the training images for use on the encoder branch. We do this as we want the encoder to
    # learn more general structures. (Use adjustments from Algorithm 1)
    augmentation_index = np.random.randint(1, 4)
    lam = np.random.uniform(0.8, 1.2)
    if augmentation_index == 1:
      augmented_images = torchvision.transforms.functional.adjust_brightness(random_image, lam)
    elif augmentation_index == 2:
      augmented_images = torchvision.transforms.functional.adjust_contrast(random_image, lam)
    elif augmentation_index == 3:
      augmented_images = torchvision.transforms.functional.adjust_saturation(random_image, lam)

    # With the augmented training images, run forward passes on the autoencoder and the teacher,
    # and normalize as we did before.
    autoencoder_output = autoencoder(augmented_images)
    with torch.no_grad():
      teacher_output_augmented = teacher(augmented_images)

    normalized_teacher_output_augmented = (teacher_output_augmented - mu) / sigma
    student_output_augmented = student(augmented_images)
    student_output_augmented_l384 = student_output_augmented[:, 384:, :, :]
    diff_ta = (normalized_teacher_output_augmented - autoencoder_output)**2
    diff_sa = (autoencoder_output - student_output_augmented_l384)**2

    # For the loss, we take the sum of the means of each of the differences (teacher-autoencoder, student-autoencoder,
    # and the student-teacher loss from earlier)
    loss_ta = torch.mean(diff_ta)
    loss_sa = torch.mean(diff_sa)
    loss = loss_st + loss_ta + loss_sa

    if i % 500 == 0:
      print("At iteration:", i, " loss: ",  loss)

    # Update the Adam optimizer parameters. Clear old gradients (avoid accumulation), and step
    sa_optimizer.zero_grad()
    loss.backward()
    sa_optimizer.step()

    # Decay learning rate after 4500 iterations
    if i == 66500:
      sa_optimizer.param_groups[0]['lr'] = 1e-4


  # Here, the training of student and autoencoder weights is done. We now make anomaly maps
  # and get anomaly scores
  st_anomaly_scores = []
  sa_anomaly_scores = []

  student.eval()
  autoencoder.eval()
  # Use more normal images from the validation set to get high end differences in normal images
  for i in range(len(validation_data)):
    val_image = validation_data[i:i+1]

    # Get outputs and normalize/compute loss similar to before
    with torch.no_grad():
      teacher_output = teacher(val_image)
    student_output = student(val_image)
    autoencoder_output = autoencoder(val_image)

    student_output_f384 = student_output[:, :384, :, :]
    student_output_l384 = student_output[:, 384:, :, :]
    normalized_teacher_output = (teacher_output - mu) / sigma
    diff_st = (normalized_teacher_output - student_output_f384)**2
    diff_sa = (autoencoder_output - student_output_l384)**2

    # Make anomaly maps using the differences. Take mean difference across channel for each output pixel
    map_st = diff_st.mean(dim=1)
    map_sa = diff_sa.mean(dim=1)

    # Resize with bilinear interpolation (like done in the autoencoder forward pass)
    map_st = F.interpolate(input=map_st.unsqueeze(1), size=(256, 256), mode='bilinear')
    map_sa = F.interpolate(input=map_sa.unsqueeze(1), size=(256, 256), mode='bilinear')

    # Append vectorized anomaly scores to score lists
    st_anomaly_scores.append(torch.flatten(map_st))
    sa_anomaly_scores.append(torch.flatten(map_sa))

  # We now have a list of flattened local (st) anomaly maps, and global (sa)
  # Get quantile values for 0.9 and 0.995 for each map so we have a reference later for what to consider an anomaly
  st_quantile_90 = torch.quantile(torch.cat(st_anomaly_scores), 0.90)
  st_quantile_995 = torch.quantile(torch.cat(st_anomaly_scores), 0.995)

  sa_quantile_90 = torch.quantile(torch.cat(sa_anomaly_scores), 0.90)
  sa_quantile_995 = torch.quantile(torch.cat(sa_anomaly_scores), 0.995)

  return teacher, student, autoencoder, mu, sigma, st_quantile_90, st_quantile_995, sa_quantile_90, sa_quantile_995

In [19]:
# Get data for training function
pasta_training_data, pasta_testing_data_np, pasta_testing_labels = load_dataset("pasta")
screws_training_data, screws_testing_data_np, screws_testing_labels = load_dataset("screws")
capsule_training_data, capsule_testing_data_np, capsule_testing_labels = load_dataset("capsule")

n_pasta = len(pasta_training_data)
n_screws = len(screws_training_data)
n_capsule = len(capsule_training_data)

val_idx_pasta = int(n_pasta*0.8)
val_idx_screws = int(n_screws*0.8)
val_idx_capsule = int(n_capsule*0.8)

pasta_train_data_np = pasta_training_data[:val_idx_pasta]
pasta_validation_data_np = pasta_training_data[val_idx_pasta:]

screws_train_data_np = screws_training_data[:val_idx_screws]
screws_validation_data_np = screws_training_data[val_idx_screws:]

capsule_train_data_np = capsule_training_data[:val_idx_capsule]
capsule_validation_data_np = capsule_training_data[val_idx_capsule:]

# Do all preprocessing once and send it to the GPU
pasta_train = image_preprocessing(pasta_train_data_np, device)
pasta_validation = image_preprocessing(pasta_validation_data_np, device)
pasta_test = image_preprocessing(pasta_testing_data_np, device)
print("pasta train length: ", len(pasta_train))
print("pasta val length: ", len(pasta_validation))
print("pasta test length: ", len(pasta_test))

screws_train = image_preprocessing(screws_train_data_np, device)
screws_validation = image_preprocessing(screws_validation_data_np, device)
screws_test = image_preprocessing(screws_testing_data_np, device)
print("screws train length: ", len(screws_train))
print("screws val length: ", len(screws_validation))
print("screws test length: ", len(screws_test))

capsule_train = image_preprocessing(capsule_train_data_np, device)
capsule_validation = image_preprocessing(capsule_validation_data_np, device)
capsule_test = image_preprocessing(capsule_testing_data_np, device)
print("capsule train length: ", len(capsule_train))
print("capsule val length: ", len(capsule_validation))
print("capsule test length: ", len(capsule_test))

student = PDN(out_channels=768).to(device)
autoencoder = AutoEncoder().to(device)

student.train()
autoencoder.train()
teacher.eval()


pasta train length:  13
pasta val length:  4
pasta test length:  16
screws train length:  14
screws val length:  4
screws test length:  15
capsule train length:  16
capsule val length:  4
capsule test length:  36


PDN(
  (conv1): Conv2d(3, 128, kernel_size=(4, 4), stride=(1, 1), padding=(3, 3))
  (avgpool1): AvgPool2d(kernel_size=2, stride=2, padding=1)
  (conv2): Conv2d(128, 256, kernel_size=(4, 4), stride=(1, 1), padding=(3, 3))
  (avgpool2): AvgPool2d(kernel_size=2, stride=2, padding=1)
  (conv3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(256, 384, kernel_size=(4, 4), stride=(1, 1))
  (relu): ReLU(inplace=True)
)

In [20]:
# --- PASTA TRAINING ---
print("Starting Pasta Training...")
t_net, s_net, ae_net, mu_p, sigma_p, st_95, st_995, sa_95, sa_995 = train_student_and_autoencoder(pasta_train, pasta_validation, pasta_test, pasta_testing_labels)

torch.save(s_net.state_dict(), '/content/drive/MyDrive/ECE471_Project/weights/pasta_student_FINAL_7000.pth')
torch.save(ae_net.state_dict(), '/content/drive/MyDrive/ECE471_Project/weights/pasta_autoencoder_FINAL_7000.pth')
metadata = {'mu': mu_p, 'sigma': sigma_p, 'st_90': st_95, 'st_995': st_995, 'sa_90': sa_95, 'sa_995': sa_995}
torch.save(metadata, '/content/drive/MyDrive/ECE471_Project/weights/pasta_metadata_FINAL_7000.pth')

Starting Pasta Training...
At iteration: 500  loss:  tensor(16.7742, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 1000  loss:  tensor(11.2783, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 1500  loss:  tensor(10.2768, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 2000  loss:  tensor(9.5719, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 2500  loss:  tensor(8.7139, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 3000  loss:  tensor(8.3668, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 3500  loss:  tensor(8.3091, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 4000  loss:  tensor(30.0802, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 4500  loss:  tensor(24.6529, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 5000  loss:  tensor(23.5874, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 5500  loss:  tensor(24.3609, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 6000  loss:  tensor(23.0648, device='cud

In [ ]:
# --- SCREWS TRAINING ---
print("Starting Screws Training...")
# We must re-define the models here so they start with fresh weights
student = PDN(out_channels=768).to(device)
autoencoder = AutoEncoder().to(device)
student.train()
autoencoder.train()

t_net, s_net, ae_net, mu_p, sigma_p, st_95, st_995, sa_95, sa_995 = train_student_and_autoencoder(screws_train, screws_validation, screws_test, screws_testing_labels)

torch.save(s_net.state_dict(), '/content/drive/MyDrive/ECE471_Project/weights/screws_student_FINAL.pth')
torch.save(ae_net.state_dict(), '/content/drive/MyDrive/ECE471_Project/weights/screws_autoencoder_FINAL.pth')
metadata = {'mu': mu_p, 'sigma': sigma_p, 'st_90': st_95, 'st_995': st_995, 'sa_90': sa_95, 'sa_995': sa_995}
torch.save(metadata, '/content/drive/MyDrive/ECE471_Project/weights/screws_metadata_FINAL.pth')

Starting Screws Training...
At iteration: 500  loss:  tensor(18.4349, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 1000  loss:  tensor(12.7657, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 1500  loss:  tensor(11.0845, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 2000  loss:  tensor(10.2260, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 2500  loss:  tensor(9.2766, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 3000  loss:  tensor(9.0459, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 3500  loss:  tensor(8.3376, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 4000  loss:  tensor(7.7624, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 4500  loss:  tensor(7.7281, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 5000  loss:  tensor(7.2495, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 5500  loss:  tensor(7.4481, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 6000  loss:  tensor(7.1191, device='cuda:0

In [ ]:
# --- CAPSULE TRAINING ---
print("Starting Capsule Training...")
student = PDN(out_channels=768).to(device)
autoencoder = AutoEncoder().to(device)
student.train()
autoencoder.train()

t_net, s_net, ae_net, mu_p, sigma_p, st_95, st_995, sa_95, sa_995 = train_student_and_autoencoder(capsule_train, capsule_validation, capsule_test, capsule_testing_labels)

torch.save(s_net.state_dict(), '/content/drive/MyDrive/ECE471_Project/weights/capsule_student_FINAL.pth')
torch.save(ae_net.state_dict(), '/content/drive/MyDrive/ECE471_Project/weights/capsule_autoencoder_FINAL.pth')
metadata = {'mu': mu_p, 'sigma': sigma_p, 'st_90': st_95, 'st_995': st_995, 'sa_90': sa_95, 'sa_995': sa_995}
torch.save(metadata, '/content/drive/MyDrive/ECE471_Project/weights/capsule_metadata_FINAL.pth')

print("All training complete!")

Starting Capsule Training...
At iteration: 500  loss:  tensor(17.6210, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 1000  loss:  tensor(12.1231, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 1500  loss:  tensor(10.3204, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 2000  loss:  tensor(9.3918, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 2500  loss:  tensor(8.8837, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 3000  loss:  tensor(8.1149, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 3500  loss:  tensor(7.5304, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 4000  loss:  tensor(7.2296, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 4500  loss:  tensor(6.6432, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 5000  loss:  tensor(6.5490, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 5500  loss:  tensor(5.9540, device='cuda:0', grad_fn=<AddBackward0>)
At iteration: 6000  loss:  tensor(5.5519, device='cuda:0

In [ ]:
# Algorithm 2 inplementation:
def inference(teacher, student, autoencoder, mu, sigma, st_quantile_95, st_quantile_995, sa_quantile_95, sa_quantile_995, test_image):
  # Get test image and get outputs of each network
  teacher_output = teacher(test_image)
  student_output = student(test_image)
  autoencoder_output = autoencoder(test_image)

  # Normalize the teacher output and get the differences as in training function
  normalized_teacher_output = (teacher_output - mu) / sigma
  student_output_f384 = student_output[:, :384, :, :]
  student_output_l384 = student_output[:, 384:, :, :]
  diff_st = (normalized_teacher_output - student_output_f384)**2
  diff_sa = (autoencoder_output - student_output_l384)**2

  # Generate anomaly maps as done in training
  map_st = diff_st.mean(dim=1)
  map_sa = diff_sa.mean(dim=1)
  map_st = F.interpolate(input=map_st.unsqueeze(1), size=(256, 256), mode='bilinear')
  map_sa = F.interpolate(input=map_sa.unsqueeze(1), size=(256, 256), mode='bilinear')

  # Normalize the original maps and combine them
  normalized_map_st = 0.1 * (map_st - st_quantile_95) / (st_quantile_995 - st_quantile_95)
  normalized_map_sa = 0.1 * (map_sa - sa_quantile_95) / (sa_quantile_995 - sa_quantile_95)

  combined_map = 0.5 * normalized_map_st + 0.5 * normalized_map_sa
  image_level_score = max(torch.flatten(combined_map))

  return combined_map, image_level_score

In [ ]:
import torch
import os

# 1. Define the current device (checks for GPU, defaults to CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WEIGHTS_PATH = '/content/drive/MyDrive/ECE471_Project/weights'

# 2. Load Metadata with map_location
# Using 'device' ensures it maps to whatever you're currently running on
meta = torch.load(os.path.join(WEIGHTS_PATH, 'pasta_metadata.pth'), map_location=device)
mu, sigma = meta['mu'], meta['sigma']

# Note: Using 'st_90' because that is the key you used to save the 95th percentile data
q95_st, q995_st = meta['st_90'], meta['st_995']
q95_sa, q995_sa = meta['sa_90'], meta['sa_995']

# 3. Load Model Weights with map_location
student.load_state_dict(torch.load(os.path.join(WEIGHTS_PATH, 'pasta_student.pth'), map_location=device))
autoencoder.load_state_dict(torch.load(os.path.join(WEIGHTS_PATH, 'pasta_autoencoder.pth'), map_location=device))

# 4. Final step: Move models to the active device and set to eval
student.to(device).eval()
autoencoder.to(device).eval()
teacher.to(device).eval()

print(f"Models successfully loaded onto {device}!")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import torch

# 1. Updated signature to accept all metadata and models
def test_on_folder(folder_path, label, teacher, student, autoencoder, mu, sigma, q95_st, q995_st, q95_sa, q995_sa, n):
    # Grab the second image in the folder (index 1)
    files = [f for f in os.listdir(folder_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    if len(files) < n:
        print(f"Not enough images in {folder_path} to use index [1]")
        return
    elif n < 1:
        print("Image index must be 1 or greater")
        return

    img_name = files[n-1]
    img_path = os.path.join(folder_path, img_name)

    # Preprocess
    img = Image.open(img_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(device)

    # Use the specific models and metadata passed in
    with torch.no_grad():
        heatmap, score = inference(
            teacher, student, autoencoder,
            mu, sigma, q95_st, q995_st, q95_sa, q995_sa,
            input_tensor
        )

    print(f"File: {img_name} | Label: {label} | Anomaly Score: {score.item():.4f}")

    # Visualization
    plt.imshow(img.resize((256, 256)))
    # The heatmap layer
    im = plt.imshow(heatmap.squeeze().cpu().numpy(), cmap='jet', alpha=0.5)

    # ADD THIS: Creates the "legend" (colorbar) on the right
    # fraction=0.046 and pad=0.04 are "magic numbers" to make the bar match the image height
    plt.colorbar(im, fraction=0.046, pad=0.04, label='Anomaly Intensity')

    plt.title(f"Anomaly Map - {label}\nScore: {score.item():.4f}")
    plt.axis('off')
    plt.show()

In [ ]:
def test_dataset(class_name, n):
    WEIGHTS_PATH = '/content/drive/MyDrive/ECE471_Project/weights/'

    # 1. Determine the current device (automatically handles CPU vs GPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 2. Load Metadata with map_location
    # This maps the "CUDA" weights to "CPU" (or vice versa) automatically
    meta = torch.load(f"{WEIGHTS_PATH}{class_name}_metadata_newteach.pth", map_location=device)

    # 3. Initialize fresh models and move them to the device
    s_net = PDN(out_channels=768).to(device)
    ae_net = AutoEncoder().to(device)

    # 4. Load Model Weights with map_location
    s_net.load_state_dict(torch.load(f"{WEIGHTS_PATH}{class_name}_student_newteach.pth", map_location=device))
    ae_net.load_state_dict(torch.load(f"{WEIGHTS_PATH}{class_name}_autoencoder_newteach.pth", map_location=device))

    s_net.eval()
    ae_net.eval()
    teacher.to(device).eval()

    print(f"--- Loaded {class_name.upper()} Model and Metadata on {device} ---")

    # Now call your test logic
    # (Using 'st_90' because that's the key you used to save the 95th percentile)
    test_on_folder(f'dataset/{class_name}/test/good', 'Normal',
                   teacher, s_net, ae_net,
                   meta['mu'], meta['sigma'],
                   meta['st_90'], meta['st_995'], meta['sa_90'], meta['sa_995'], n)

    test_on_folder(f'dataset/{class_name}/test/bad', 'Anomalous',
                   teacher, s_net, ae_net,
                   meta['mu'], meta['sigma'],
                   meta['st_90'], meta['st_995'], meta['sa_90'], meta['sa_995'], n)

In [ ]:
test_dataset("screws", 5)

In [ ]:
# TODO use your class above as well as helper functions to generate your
# predictions on the datasets and evaluate the results.
def do_analysis(ad, class_name):
  training_images, testing_images, testing_labels = load_dataset(class_name=class_name)
  ad.create_model(training_images)
  predictions = ad.predict(testing_images)
  basic_evaluation(predictions, testing_labels)

do_analysis(AnomalyDetector(), 'screws')
do_analysis(AnomalyDetector(), 'pasta')
